In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS helpdesk_bronze;
CREATE SCHEMA IF NOT EXISTS helpdesk_silver;
CREATE SCHEMA IF NOT EXISTS helpdesk_gold;

In [0]:
from pyspark.sql import functions as F

volume_path = "/Volumes/workspace/default/dados_chamados/"

# Função para substituir espaços por _ nos nomes das colunas
def limpa_colunas(df):
    for c in df.columns:
        novo_nome = c.replace(" ", "_").replace("(", "").replace(")", "").replace(";", "").replace(",", "")
        df = df.withColumnRenamed(c, novo_nome)
    return df

# Leitura dos ficheiros novos _2
df_f_raw = spark.read.csv(f"{volume_path}Base_Fechados.csv", header=True, sep=";")
df_r_raw = spark.read.csv(f"{volume_path}Base_Reabertura.csv", header=True, sep=";")
df_s_raw = spark.read.csv(f"{volume_path}Base_SLA.csv", header=True, sep=";")

# Aplica a limpeza
df_f_raw = limpa_colunas(df_f_raw)
df_r_raw = limpa_colunas(df_r_raw)
df_s_raw = limpa_colunas(df_s_raw)

# Adiciona metadados
df_fechados_bronze = df_f_raw.withColumn("_data_ingestao", F.current_timestamp())
df_reabertura_bronze = df_r_raw.withColumn("_data_ingestao", F.current_timestamp())
df_sla_bronze = df_s_raw.withColumn("_data_ingestao", F.current_timestamp())

# Salva em Delta
df_fechados_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("helpdesk_bronze.raw_fechados")
df_reabertura_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("helpdesk_bronze.raw_reabertura")
df_sla_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("helpdesk_bronze.raw_sla")

print("Camada Bronze persistida com sucesso!")

Camada Bronze persistida com sucesso!


In [0]:
from pyspark.sql import functions as F

df_f = spark.table("helpdesk_bronze.raw_fechados")
df_r = spark.table("helpdesk_bronze.raw_reabertura")
df_s = spark.table("helpdesk_bronze.raw_sla")

df_f_clean = df_f.select(
    F.col("Solicitação_de_Serviço").alias("id_chamado"),
    F.col("Prioridade_Interna").alias("prioridade"),
    F.col("Grupo_de_Proprietários_Designado").alias("grupo_tecnico"),
    F.to_date(F.col("Relatado_Em"), "dd/MM/yy").alias("data_abertura"),
    F.to_date(F.col("Término_Efetivo"), "dd/MM/yy").alias("data_fechamento")
).dropDuplicates(["id_chamado"])

df_r_clean = df_r.select(
    F.col("Solicitação_de_serviço").alias("id_chamado"),
    F.col("Quantidade_de_reaberturas").cast("int").alias("qtd_reaberturas")
).dropDuplicates(["id_chamado"])

df_s_clean = df_s.select(
    F.col("Solicitação_de_serviço").alias("id_chamado"),
    F.col("Tempo_de_atendimento").alias("tempo_atendimento_str")
).dropDuplicates(["id_chamado"])

# Unificação das tabelas
df_silver = df_f_clean.join(df_r_clean, on="id_chamado", how="left") \
                      .join(df_s_clean, on="id_chamado", how="left")

# Regra: converter "HH:mm" para tempo decimal
df_silver = df_silver.withColumn(
    "qtd_reaberturas", F.coalesce(F.col("qtd_reaberturas"), F.lit(0))
).withColumn(
    "reaberto_flag", F.when(F.col("qtd_reaberturas") > 0, 1).otherwise(0)
).withColumn(
    "tempo_atendimento_horas",
    F.round(
        F.split(F.col("tempo_atendimento_str"), ":").getItem(0).cast("double") + 
        (F.split(F.col("tempo_atendimento_str"), ":").getItem(1).cast("double") / 60.0), 
        2
    )
)

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("helpdesk_silver.chamados_consolidados")
print("Camada Silver persistida com sucesso!")

Camada Silver persistida com sucesso!


In [0]:
%sql
CREATE OR REPLACE TABLE helpdesk_gold.dim_grupo AS
SELECT ROW_NUMBER() OVER (ORDER BY grupo_tecnico) AS id_grupo, grupo_tecnico AS nome_grupo
FROM (SELECT DISTINCT grupo_tecnico FROM helpdesk_silver.chamados_consolidados WHERE grupo_tecnico IS NOT NULL);

CREATE OR REPLACE TABLE helpdesk_gold.dim_tempo AS
SELECT 
    INT(DATE_FORMAT(data_abertura, 'yyyyMMdd')) AS id_tempo, data_abertura AS data_completa,
    YEAR(data_abertura) AS ano, MONTH(data_abertura) AS mes, DATE_FORMAT(data_abertura, 'EEEE') AS dia_semana
FROM (SELECT DISTINCT data_abertura FROM helpdesk_silver.chamados_consolidados WHERE data_abertura IS NOT NULL);

-- Tabela Fato com Regras de Negócio de SLA
CREATE OR REPLACE TABLE helpdesk_gold.fato_chamados AS
SELECT 
    s.id_chamado,
    dg.id_grupo,
    INT(DATE_FORMAT(s.data_abertura, 'yyyyMMdd')) AS id_tempo_abertura,
    s.prioridade,
    s.reaberto_flag,
    s.tempo_atendimento_horas,
    
    -- REGRA: Limite de Horas por Prioridade
    CASE 
        WHEN s.prioridade = '1' THEN 1.0
        WHEN s.prioridade = '2' THEN 2.0
        WHEN s.prioridade = '3' THEN 4.0
        WHEN s.prioridade = '4' THEN 8.0
        ELSE NULL
    END AS sla_horas_limite,
    
    -- REGRA: Cumprimento do SLA
    CASE 
        WHEN s.tempo_atendimento_horas <= (
            CASE 
                WHEN s.prioridade = '1' THEN 1.0
                WHEN s.prioridade = '2' THEN 2.0
                WHEN s.prioridade = '3' THEN 4.0
                WHEN s.prioridade = '4' THEN 8.0
            END
        ) THEN 1 ELSE 0 
    END AS sla_cumprido_flag
    
FROM helpdesk_silver.chamados_consolidados s
LEFT JOIN helpdesk_gold.dim_grupo dg ON s.grupo_tecnico = dg.nome_grupo;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT 
    f.prioridade,
    dg.nome_grupo AS grupo_solucionador,
    COUNT(f.id_chamado) AS total_chamados,
    ROUND(AVG(f.tempo_atendimento_horas), 2) AS tempo_medio_horas,
    
    -- Análise de Reabertura (Meta: < 5%)
    ROUND(AVG(f.reaberto_flag) * 100, 2) AS taxa_reabertura_pct,
    CASE WHEN AVG(f.reaberto_flag) * 100 < 5.0 THEN 'DENTRO DA META' ELSE 'FORA DA META' END AS meta_reabertura,
    
    -- Análise de SLA (Meta: >= 85%)
    ROUND(AVG(f.sla_cumprido_flag) * 100, 2) AS taxa_sla_cumprido_pct,
    CASE WHEN AVG(f.sla_cumprido_flag) * 100 >= 85.0 THEN 'DENTRO DA META' ELSE 'FORA DA META' END AS meta_sla

FROM helpdesk_gold.fato_chamados f
LEFT JOIN helpdesk_gold.dim_grupo dg ON f.id_grupo = dg.id_grupo
WHERE f.prioridade IN ('1', '2', '3', '4')
GROUP BY f.prioridade, dg.nome_grupo
ORDER BY f.prioridade ASC, dg.nome_grupo ASC;

prioridade,grupo_solucionador,total_chamados,tempo_medio_horas,taxa_reabertura_pct,meta_reabertura,taxa_sla_cumprido_pct,meta_sla
1,N1,155,17.21,0.0,DENTRO DA META,76.77,FORA DA META
1,N2-A,98,3.1,0.0,DENTRO DA META,76.53,FORA DA META
1,N2-RIO,127,2.56,0.0,DENTRO DA META,78.74,FORA DA META
2,N2-RIO,1,1.2,0.0,DENTRO DA META,100.0,DENTRO DA META
3,N1,194,95.28,0.0,DENTRO DA META,51.03,FORA DA META
3,N2-A,77,1.46,5.19,FORA DA META,92.21,DENTRO DA META
3,N2-RIO,15,7.04,0.0,DENTRO DA META,86.67,DENTRO DA META
4,N1,2436,18.25,0.86,DENTRO DA META,80.09,FORA DA META
4,N2-A,2751,6.4,0.73,DENTRO DA META,85.53,DENTRO DA META
4,N2-RIO,1149,7.18,1.48,DENTRO DA META,82.25,FORA DA META


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    dt.dia_semana,
    COUNT(f.id_chamado) AS volume_aberturas,
    ROUND(AVG(f.tempo_atendimento_horas), 2) AS tempo_medio_resolucao_horas,
    ROUND(AVG(f.sla_cumprido_flag) * 100, 2) AS taxa_sla_cumprido_pct
FROM helpdesk_gold.fato_chamados f
INNER JOIN helpdesk_gold.dim_tempo dt ON f.id_tempo_abertura = dt.id_tempo
GROUP BY dt.dia_semana
ORDER BY volume_aberturas DESC;

dia_semana,volume_aberturas,tempo_medio_resolucao_horas,taxa_sla_cumprido_pct
Monday,1631,9.39,80.5
Tuesday,1624,11.23,81.47
Wednesday,1443,12.3,80.87
Thursday,1281,20.85,80.64
Friday,1104,12.32,76.36
Saturday,40,17.08,60.0
Sunday,39,21.58,61.54


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
COMMENT ON TABLE helpdesk_gold.fato_chamados IS 'Tabela fato central de eventos e métricas de chamados.';
COMMENT ON COLUMN helpdesk_gold.fato_chamados.sla_cumprido_flag IS 'Indicador: 1 se cumpriu prazo (P1=1h, P2=2h, P3=4h, P4=8h), 0 se excedeu.';
COMMENT ON COLUMN helpdesk_gold.fato_chamados.reaberto_flag IS 'Indicador: 1 se foi reaberto, 0 caso contrario.';